# Convolutional Neural Networks (CNNs) with TensorFlow/Keras

This notebook covers:
- Building CNNs from scratch with TensorFlow/Keras
- Binary image classification (pizza vs steak)
- Multi-class image classification (10 food classes)
- Data augmentation techniques
- The Tiny VGG architecture
- Model evaluation and prediction

## What are CNNs?

**Convolutional Neural Networks (CNNs)** are specialized neural networks for image data.

### Key Components:

1. **Convolutional Layers (Conv2D)**
   - Extract features from images
   - Learn filters/kernels automatically
   - Preserve spatial information

2. **Pooling Layers (MaxPool2D)**
   - Reduce spatial dimensions
   - Make features translation-invariant
   - Reduce computation

3. **Flatten Layer**
   - Convert 2D features → 1D vector
   - Connect to Dense layers

4. **Dense Layers**
   - Final classification
   - Learn combinations of features

### CNN Architecture Pattern:
```
Input Image (224×224×3)
    ↓
Conv2D → ReLU → Conv2D → ReLU → MaxPool2D  (Block 1)
    ↓
Conv2D → ReLU → Conv2D → ReLU → MaxPool2D  (Block 2)
    ↓
Flatten
    ↓
Dense → Output
```

## Datasets:
1. **Pizza vs Steak**: Binary classification
2. **10 Food Classes**: Multi-class classification

## Setup and Imports

In [ ]:
# Print timestamp for tracking when notebook was run
import datetime
print(f"Notebook last run (end-to-end): {datetime.datetime.now()}")

## Part 1: Download and Prepare Data

In [ ]:
import urllib.request
import zipfile
import os

# Download pizza vs steak dataset
url = "https://storage.googleapis.com/ztm_tf_course/food_vision/pizza_steak.zip"
zip_path = "pizza_steak.zip"
extract_dir = "pizza_steak"

# Check if already downloaded and extracted
if not os.path.exists(extract_dir):
    print(f"Downloading {url}...")
    
    # Download file
    urllib.request.urlretrieve(url, zip_path)
    print(f"Downloaded to {zip_path}")
    
    # Extract zip file
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    
    # Remove zip file to save space
    os.remove(zip_path)
    print(f"Extracted to {extract_dir}/ and removed {zip_path}")
else:
    print(f"Dataset already exists at {extract_dir}/")

In [ ]:
def print_directory_tree(path, prefix="", max_files=3):
    """
    Print directory structure with limited files per folder.
    Helps visualize dataset organization.
    
    Args:
        path: Directory path to print
        prefix: Prefix for tree structure visualization
        max_files: Maximum number of files to show per folder
    """
    try:
        entries = sorted(os.listdir(path))
        dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
        files = [e for e in entries if os.path.isfile(os.path.join(path, e))]
        
        # Print directories
        for i, d in enumerate(dirs):
            is_last = (i == len(dirs) - 1) and len(files) == 0
            print(f"{prefix}{'└── ' if is_last else '├── '}{d}/")
            
            # Recursively print subdirectories
            new_prefix = prefix + ("    " if is_last else "│   ")
            print_directory_tree(os.path.join(path, d), new_prefix, max_files)
        
        # Print files (limited to max_files)
        for i, f in enumerate(files[:max_files]):
            is_last = i == min(len(files), max_files) - 1
            print(f"{prefix}{'└── ' if is_last else '├── '}{f}")
        
        # Show count if more files exist
        if len(files) > max_files:
            print(f"{prefix}    ... ({len(files) - max_files} more files)")
            
    except PermissionError:
        print(f"{prefix}[Permission Denied]")

# Print the dataset structure
print("\nDataset Structure:")
print("pizza_steak/")
print_directory_tree("pizza_steak")

## Part 2: Explore the Data

In [ ]:
# Get class names programmatically
# This is better than hardcoding, especially with many classes
import pathlib
import numpy as np

data_dir = pathlib.Path("pizza_steak/train/")

# Get all subdirectory names (these are our classes)
class_names = np.array(
    sorted([item.name for item in data_dir.glob('*')])
)

print(f"Class names: {class_names}")
print(f"Number of classes: {len(class_names)}")

In [ ]:
# Function to visualize random images
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

def view_random_image(target_dir, target_class):
    """
    Display a random image from a target directory and class.
    
    Args:
        target_dir: Directory containing class folders
        target_class: Class name to select from
        
    Returns:
        Image array
    """
    # Build path to target class folder
    target_folder = target_dir + target_class
    
    # Get random image from folder
    random_image = random.sample(os.listdir(target_folder), 1)[0]
    
    # Read and display image
    img = mpimg.imread(target_folder + "/" + random_image)
    plt.imshow(img)
    plt.title(target_class)
    plt.axis("off")
    
    print(f"Image shape: {img.shape}")  # (height, width, color_channels)
    print(f"Image path: {target_folder}/{random_image}")
    
    return img

In [ ]:
# View a random pizza image
img = view_random_image(
    target_dir="pizza_steak/train/",
    target_class="pizza"
)

In [ ]:
# Check image shape
img.shape  # Returns (height, width, color_channels)

# Typical shape: (height, width, 3)
# 3 color channels = RGB (Red, Green, Blue)

## Part 3: Prepare Data with ImageDataGenerator

### ImageDataGenerator Benefits:
- Loads images from directories automatically
- Rescales pixel values (0-255 → 0-1)
- Creates batches for training
- Can apply data augmentation

### Expected Directory Structure:
```
pizza_steak/
├── train/
│   ├── pizza/
│   │   ├── image1.jpg
│   │   └── image2.jpg
│   └── steak/
│       ├── image1.jpg
│       └── image2.jpg
└── test/
    ├── pizza/
    └── steak/
```

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Set random seed for reproducibility
SEED = datetime.datetime.now().microsecond
tf.random.set_seed(SEED)

# Create ImageDataGenerator instances
# rescale=1./255 converts pixel values from [0, 255] to [0, 1]
# Neural networks work better with smaller values
train_datagen = ImageDataGenerator(rescale=1.0/255.0)
valid_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Setup paths
train_dir = "pizza_steak/train/"
test_dir = "pizza_steak/test/"

# Load and prepare training data
train_data = train_datagen.flow_from_directory(
    train_dir,
    batch_size=32,              # Process 32 images at a time
    target_size=(224, 224),     # Resize all images to 224×224
    class_mode="binary",        # Binary classification (2 classes)
    seed=SEED
)

# Load and prepare test data
test_data = valid_datagen.flow_from_directory(
    test_dir,
    batch_size=32,
    target_size=(224, 224),
    class_mode="binary",
    seed=SEED
)

# ImageDataGenerator automatically:
# 1. Finds subdirectories (pizza, steak)
# 2. Labels images based on subdirectory
# 3. Creates batches
# 4. Applies rescaling

## Part 4: Build CNN Model (Tiny VGG)

### Tiny VGG Architecture:
Inspired by the famous VGG network, but smaller.

**Pattern: [Conv → Conv → MaxPool] × 2 → Flatten → Dense**

### Layer Explanations:

**Conv2D(filters=10, kernel_size=3)**
- Creates 10 different 3×3 filters
- Each filter detects different features
- Early layers: edges, corners
- Later layers: complex patterns

**MaxPool2D(pool_size=2)**
- Takes 2×2 region, keeps maximum value
- Reduces spatial dimensions by half
- Makes model translation-invariant

**Flatten()**
- Converts 3D tensor → 1D vector
- Example: (7, 7, 10) → 490

**Dense(1, activation='sigmoid')**
- Output layer for binary classification
- Sigmoid outputs probability [0, 1]
- > 0.5 = pizza, < 0.5 = steak

In [ ]:
# Create CNN model (Tiny VGG architecture)
# Reference: https://poloclub.github.io/cnn-explainer/

model_1 = tf.keras.models.Sequential([
    
    # First convolutional block
    tf.keras.layers.Conv2D(
        filters=10,              # Number of filters to learn
        kernel_size=3,           # 3×3 filter size
        activation="relu",       # ReLU activation (max(0, x))
        input_shape=(224, 224, 3)  # Input: 224×224 RGB images
    ),
    tf.keras.layers.Conv2D(
        filters=10,
        kernel_size=3,
        activation="relu"
    ),
    tf.keras.layers.MaxPool2D(
        pool_size=2,             # 2×2 pooling window
        padding="valid"          # No padding (reduces size)
    ),
    
    # Second convolutional block
    tf.keras.layers.Conv2D(
        filters=10,
        kernel_size=3,
        activation="relu"
    ),
    tf.keras.layers.Conv2D(
        filters=10,
        kernel_size=3,
        activation="relu"
    ),
    tf.keras.layers.MaxPool2D(
        pool_size=2
    ),
    
    # Flatten and output layers
    tf.keras.layers.Flatten(),   # Convert 2D → 1D
    tf.keras.layers.Dense(
        units=1,                  # 1 output (binary classification)
        activation="sigmoid"      # Sigmoid for probability [0, 1]
    )
])

# Compile the model
model_1.compile(
    loss="binary_crossentropy",   # Loss for binary classification
    optimizer=tf.keras.optimizers.Adam(),  # Adam optimizer
    metrics=["accuracy"]          # Track accuracy during training
)

In [ ]:
# Check model architecture
model_1.summary()

# Key observations:
# - Spatial dimensions decrease: 224 → 110 → 53 → 24 → 10
# - Number of parameters to train
# - Output shape at each layer

In [ ]:
# Train the model
history_1 = model_1.fit(
    train_data,
    epochs=5,
    steps_per_epoch=len(train_data),      # Number of batches per epoch
    validation_data=test_data,
    validation_steps=len(test_data)
)

## Part 5: Evaluate Model Performance

In [ ]:
# Plot training history
import pandas as pd

# Convert history to DataFrame for easy plotting
pd.DataFrame(history_1.history).plot(figsize=(10, 7))
plt.title("Training Metrics")
plt.xlabel("Epoch")
plt.ylabel("Metric Value")
plt.show()

# Look for:
# - Decreasing loss (good)
# - Increasing accuracy (good)
# - Gap between train and validation (overfitting if large)

In [ ]:
def plot_loss_curves(history):
    """
    Returns separate loss curves for training and validation metrics.
    
    Helps identify:
    - Overfitting: train acc >> val acc
    - Underfitting: both accuracies low
    - Good fit: both accuracies high and close
    """
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    
    accuracy = history.history['accuracy']
    val_accuracy = history.history['val_accuracy']
    
    epochs = range(len(history.history['loss']))
    
    # Plot loss
    plt.figure(figsize=(14, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label='Training Loss')
    plt.plot(epochs, val_loss, label='Validation Loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label='Training Accuracy')
    plt.plot(epochs, val_accuracy, label='Validation Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot loss curves
plot_loss_curves(history_1)

## Part 6: Data Augmentation

### Why Data Augmentation?

**Problem**: Limited training data can cause overfitting

**Solution**: Create variations of existing images

### Augmentation Techniques:

1. **Rotation** (rotation_range=30)
   - Rotate images up to 30°
   - Helps with tilted photos

2. **Shear** (shear_range=0.2)
   - Slant transformation
   - Simulates different viewing angles

3. **Zoom** (zoom_range=0.2)
   - Random zoom in/out
   - Handles different distances

4. **Shift** (width/height_shift_range=0.2)
   - Move image horizontally/vertically
   - Object doesn't need to be centered

5. **Flip** (horizontal_flip=True)
   - Mirror image horizontally
   - Doubles effective dataset size

### Important Notes:
- **Only augment training data**, not test data!
- Labels don't change (pizza stays pizza)
- Applied on-the-fly during training
- Different augmentation each epoch

In [ ]:
# Create augmented data generator
train_datagen_augmented = ImageDataGenerator(
    rescale=1/255.,              # Normalize pixel values
    rotation_range=30,           # Rotate ±30 degrees
    shear_range=0.2,             # Shear transformation
    zoom_range=0.2,              # Zoom in/out by 20%
    width_shift_range=0.2,       # Shift horizontally by 20%
    height_shift_range=0.2,      # Shift vertically by 20%
    horizontal_flip=True         # Random horizontal flip
)

# Test data: NO augmentation (just rescale)
# We want consistent test conditions
test_datagen = ImageDataGenerator(rescale=1/255.)

print("✓ Data augmentation configured")
print("Training data: Augmented (rotation, zoom, flip, etc.)")
print("Test data: Original (rescaled only)")

In [ ]:
# Create augmented training data
print("Augmented training images:")
train_data_augmented = train_datagen_augmented.flow_from_directory(
    train_dir,
    batch_size=32,
    target_size=(224, 224),
    class_mode="binary",
    shuffle=False  # Don't shuffle initially (for visualization)
)

print("\nTest images (no augmentation):")
test_data = test_datagen.flow_from_directory(
    test_dir,
    batch_size=32,
    target_size=(224, 224),
    class_mode="binary",
    shuffle=False
)

In [ ]:
# Visualize augmentation (uncomment to see)
# Get batch of images
# images, labels = train_data.next()  # Original images
# augmented_images, augmented_labels = train_data_augmented.next()  # Augmented

# Note: Labels stay the same (pizza is still pizza after rotation!)

In [ ]:
# Compare original vs augmented (uncomment to visualize)
# random_number = random.randint(0, 31)  # Batch size is 32

# # Original image
# plt.imshow(images[random_number])
# plt.title(f"Original image")
# plt.axis(False)

# # Augmented version
# plt.figure()
# plt.imshow(augmented_images[random_number])
# plt.title(f"Augmented image")
# plt.axis(False)

In [ ]:
# Create augmented data with shuffling for training
# Shuffling is important for better training
train_data_augmented_shuffled = train_datagen_augmented.flow_from_directory(
    train_dir,
    batch_size=32,
    target_size=(224, 224),
    class_mode="binary",
    shuffle=True  # Shuffle for training
)

## Part 7: Train Model with Augmented Data

In [ ]:
Create model with augmented data (uncomment to train)
Same architecture as model_1, but trained on augmented data

model_2 = tf.keras.models.Sequential([
    
    # First block
    tf.keras.layers.Conv2D(
        filters=16,  # Slightly more filters than model_1
        kernel_size=3,
        activation="relu",
        input_shape=(224, 224, 3)
    ),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPool2D(2),
    
    # Second block
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPool2D(2),
    
    # Output
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_2.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"]
)

# Train on augmented data
history_2 = model_2.fit(
    train_data_augmented_shuffled,  # Augmented training data
    epochs=5,
    steps_per_epoch=len(train_data_augmented_shuffled),
    validation_data=test_data,  # Original test data (no augmentation)
    validation_steps=len(test_data)
)

In [ ]:
# Plot training curves for augmented model
plot_loss_curves(history_2)

# Expected observations:
# - Training might be slightly slower (augmentation overhead)
# - Training accuracy might be lower (harder to learn)
# - Validation accuracy often BETTER (less overfitting)
# - Smaller gap between train and validation

## Part 8: Making Predictions

In [ ]:
from tensorflow.keras.preprocessing import image

def predict_image(model, img_path, class_names=['pizza', 'steak']):
    """
    Load and predict an image with proper preprocessing.
    
    Args:
        model: Trained Keras model
        img_path: Path to image file
        class_names: List of class names
    """
    # Load image with target size (MUST match training size!)
    img = image.load_img(img_path, target_size=(224, 224))
    
    # Convert to array
    img_array = image.img_to_array(img)
    
    # Add batch dimension: (224, 224, 3) → (1, 224, 224, 3)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Rescale (same as training!)
    img_array = img_array / 255.0
    
    # Make prediction
    prediction = model.predict(img_array)
    
    # For binary classification:
    # prediction[0][0] is probability of class 1
    prob = prediction[0][0]
    
    # Interpret prediction
    if prob > 0.5:
        pred_class = class_names[1]  # steak
        confidence = prob
    else:
        pred_class = class_names[0]  # pizza
        confidence = 1 - prob
    
    # Display
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.title(f"Prediction: {pred_class} ({confidence:.2%} confidence)")
    plt.axis('off')
    plt.show()
    
    print(f"Predicted: {pred_class}")
    print(f"Confidence: {confidence:.2%}")
    print(f"Raw prediction: {prob:.4f}")

## Part 9: Multi-Class Classification (10 Food Classes)

Now let's scale up from 2 classes to 10 classes!

In [ ]:
# Download 10 food classes dataset
url = "https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_all_data.zip"
zip_path = "10_food_classes_all_data.zip"
extract_dir = "10_food_classes_all_data"

# Check if already downloaded
if not os.path.exists(extract_dir):
    print(f"Downloading {url}...")
    urllib.request.urlretrieve(url, zip_path)
    
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    
    os.remove(zip_path)
    print(f"Dataset ready at {extract_dir}/")
else:
    print(f"Dataset already exists at {extract_dir}/")

In [ ]:
# Explore 10 food classes dataset
for dirpath, dirnames, filenames in os.walk("10_food_classes_all_data"):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

In [ ]:
# Setup directories
train_dir = "10_food_classes_all_data/train/"
test_dir = "10_food_classes_all_data/test/"

# Get class names for multi-class dataset
data_dir = pathlib.Path(train_dir)
class_names = np.array(
    sorted([item.name for item in data_dir.glob('*')])
)

print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")

In [ ]:
# Visualize random image from random class
img = view_random_image(
    target_dir=train_dir,
    target_class=random.choice(class_names)
)

### Prepare Multi-Class Data

**Key Difference from Binary:**
- `class_mode="categorical"` instead of `"binary"`
- One-hot encoding for labels
- Output layer: 10 units with softmax

In [ ]:
# Create data generators
train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)

# Load data
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",  # Multi-class classification
    shuffle=True
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",  # Multi-class classification
    shuffle=False
)

print(f"\nClass indices: {train_data.class_indices}")

### Build Multi-Class CNN Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense

# Similar architecture to binary model
# Key difference: Output layer has 10 units with softmax
model_3 = Sequential([
    
    # First block
    Conv2D(16, 3, activation='relu', input_shape=(224, 224, 3)),
    Conv2D(16, 3, activation='relu'),
    MaxPool2D(),
    
    # Second block
    Conv2D(16, 3, activation='relu'),
    Conv2D(16, 3, activation='relu'),
    MaxPool2D(),
    
    # Output
    Flatten(),
    Dense(
        10,                    # 10 output units (one per class)
        activation='softmax'   # Softmax for multi-class probabilities
    )
])

# Compile for multi-class classification
model_3.compile(
    loss='categorical_crossentropy',  # Multi-class loss
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

print("✓ Multi-class CNN model created")
print(f"  - Input: 224×224×3 images")
print(f"  - Output: 10 classes (softmax)")
print(f"  - Loss: categorical_crossentropy")

In [ ]:
# Train multi-class model
history_3 = model_3.fit(
    train_data,
    epochs=5,
    steps_per_epoch=len(train_data),
    validation_data=test_data,
    validation_steps=len(test_data)
)

# Note: 10 classes is harder than 2!
# Random guessing accuracy = 10% (vs 50% for binary)
# May need more training or transfer learning

In [ ]:
# Plot training curves
plot_loss_curves(history_3)

# Observations:
# - Lower accuracy than binary classification (expected)
# - May need more epochs, data augmentation, or transfer learning

In [ ]:
# This model needs improvement!
# Strategies:
# 1. More epochs (10-20+)
# 2. Data augmentation
# 3. Transfer learning (pretrained models)
# 4. More complex architecture
# 5. Learning rate scheduling

In [ ]:
# Make predictions on test images
# predict_image(model_3, "test_fig_1.jpg")
# predict_image(model_3, "test_fig_2.jpg")
# predict_image(model_3, "test_fig_3.jpg")
# predict_image(model_3, "test_fig_4.jpg")

## Part 10: Save Models

In [ ]:
# Save models for later use
# TensorFlow SavedModel format (recommended)

# Save binary classification model
# model_2.save("steak_pizza_model_2")

# Save multi-class model
# model_3.save("multiclass_model_1")

print("Models saved!")
print("\nTo load:")
print("  model = tf.keras.models.load_model('steak_pizza_model_2')")

## Summary: CNNs with TensorFlow/Keras

### Key Learnings:

**1. CNN Architecture**
```python
Sequential([
    Conv2D → Conv2D → MaxPool2D,  # Extract features
    Conv2D → Conv2D → MaxPool2D,  # Higher-level features
    Flatten,                       # 2D → 1D
    Dense                          # Classification
])
```

**2. Binary vs Multi-Class**

| Aspect | Binary | Multi-Class |
|--------|--------|-------------|
| **class_mode** | "binary" | "categorical" |
| **Output units** | 1 | num_classes |
| **Activation** | sigmoid | softmax |
| **Loss** | binary_crossentropy | categorical_crossentropy |

**3. Data Augmentation**
- Prevents overfitting
- Creates variations of training data
- Apply ONLY to training data
- Includes: rotation, zoom, flip, shift

**4. ImageDataGenerator**
- Loads images from directories
- Automatic rescaling
- Batch creation
- On-the-fly augmentation

**5. Model Evaluation**
- Plot loss curves
- Check train vs validation gap
- Large gap = overfitting
- Both low = underfitting

**6. Improvements for Better Performance**
- More data
- Data augmentation
- Deeper networks
- Transfer learning
- More training epochs
- Learning rate tuning

### Next Steps:
- Transfer learning (use pretrained models)
- More complex architectures (ResNet, EfficientNet)
- Advanced augmentation (mixup, cutout)
- Hyperparameter tuning